# Daily Challenge — Build a Retrieval Augmented Generation (RAG) System
**Week 7 · Day 3**

In this challenge we build a functional **RAG** pipeline with **LangChain** + **Hugging Face** that answers
questions grounded in a real dataset (`databricks/databricks-dolly-15k`).

**Pipeline overview**

1. **Setup** — install the libraries.
2. **Load** the dataset as LangChain `Document`s.
3. **Split** documents into overlapping chunks.
4. **Embed** the chunks with a sentence-transformer.
5. **Index** the embeddings in a **FAISS** vector store.
6. **Prepare** a pre-trained QA model from Hugging Face.
7. **Retrieve + Answer** — connect the retriever to the model.
8. **Test** the system on a query.

> ℹ️ **Note on Step 6–7.** The challenge uses `Intel/dynamic_tinybert`, which is an **extractive** QA model
> (it selects an answer *span* from a given context). LangChain's `HuggingFacePipeline` / `RetrievalQA` only
> accept **generative** LLMs (`text-generation` / `text2text-generation`), so plugging an extractive QA
> pipeline into `RetrievalQA` raises a `ValueError`. We therefore wire retrieval → extractive-QA directly
> (the correct pattern for this model), and add a **bonus** `RetrievalQA` chain using a generative model.


## 1) Set up your environment
Install everything the RAG system needs: LangChain (orchestration), Transformers (models),
sentence-transformers (embeddings), datasets (data), and FAISS (similarity search).

In [ ]:
!pip -q install -U datasets transformers sentence-transformers faiss-cpu \
    langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface

In [ ]:
# Imports (robust across LangChain versions)
from typing import List

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    pipeline,
)

from langchain_core.documents import Document

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

print("Imports OK")

## 2) Load the dataset
`databricks/databricks-dolly-15k` has columns `instruction`, `context`, `response`, `category`.
We use the **`context`** column as the retrievable text.

Many rows have an **empty** `context`, so we load the split directly and keep only non-empty contexts
(this is cleaner and faster than indexing thousands of blank documents).

In [ ]:
dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

ds = load_dataset(dataset_name, split="train")
print("Raw rows:", len(ds))

# Keep only rows that actually have context text, and wrap them as LangChain Documents.
data: List[Document] = []
for row in ds:
    ctx = (row.get(page_content_column) or "").strip()
    if not ctx:
        continue
    data.append(
        Document(
            page_content=ctx,
            metadata={
                "instruction": row.get("instruction", ""),
                "category": row.get("category", ""),
            },
        )
    )

print("Documents with non-empty context:", len(data))
print("\nExample document:\n", data[0].page_content[:300])

## 3) Split the documents
LLMs and embedders have a limited context window. `RecursiveCharacterTextSplitter` breaks long
documents into overlapping chunks so no context is lost at the boundaries.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

docs = text_splitter.split_documents(data)
print("Chunks:", len(docs))
print("\nFirst chunk:\n", docs[0].page_content[:300])

## 4) Embed the text
We turn each chunk into a numerical vector that captures its **semantic meaning**, using the
`all-MiniLM-L6-v2` sentence-transformer. Similar texts end up close together in vector space.

In [ ]:
modelPath = "sentence-transformers/all-MiniLM-l6-v2"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": False}

embeddings = HuggingFaceEmbeddings(
    model_name=modelPath,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

# Quick sanity check
query_result = embeddings.embed_query("This is a test document.")
print("Embedding dimension:", len(query_result))
print("First 3 values:", query_result[:3])

## 5) Create a vector store (FAISS)
FAISS indexes the chunk embeddings so we can run **fast similarity search** at query time.

> ⏳ Building the index embeds every chunk — this can take a few minutes depending on how many
> chunks you kept.

In [ ]:
db = FAISS.from_documents(
    docs,
    embeddings,
    distance_strategy=DistanceStrategy.COSINE,
)
print("FAISS index built. Vectors:", db.index.ntotal)

## 6) Prepare the QA model
We load `Intel/dynamic_tinybert`, a fine-tuned **extractive** question-answering model. Given a
`question` and a `context`, it returns the answer **span** found inside that context (plus a
confidence score).

In [ ]:
model_name = "Intel/dynamic_tinybert"

tokenizer = AutoTokenizer.from_pretrained(
    model_name, padding=True, truncation=True, max_length=512
)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

question_answerer = pipeline(
    "question-answering",
    model=model,
    tokenizer=tokenizer,
)

# Smoke test the pipeline on a tiny hand-made context
demo = question_answerer(
    question="Where is the Eiffel Tower?",
    context="The Eiffel Tower is a wrought-iron lattice tower located in Paris, France.",
)
print(demo)

## 7) Build the Retrieval + QA pipeline
The **retriever** finds the most relevant chunks from FAISS; the **QA model** extracts the answer
from those chunks.

Because `Intel/dynamic_tinybert` is *extractive* (not generative), we do **not** use
`RetrievalQA.from_chain_type` here — that chain expects a generative LLM. Instead we retrieve the
top-k chunks, join them into a single context, and let the extractive QA model pull out the answer.
This is the correct way to use this model in a RAG setup.

In [ ]:
retriever = db.as_retriever(search_kwargs={"k": 4})

def rag_answer(question: str, k: int = 4):
    """Retrieve top-k chunks from FAISS and extract an answer with the QA model."""
    docs = retriever.invoke(question)[:k]
    context = "\n\n".join(d.page_content for d in docs)

    result = question_answerer(question=question, context=context)
    return {
        "question": question,
        "answer": result["answer"],
        "score": result["score"],
        "source_documents": docs,
    }

print("RAG pipeline ready")

## 8) Test your RAG system
Ask a question. The system retrieves relevant Dolly contexts and extracts an answer from them.

In [ ]:
question = "What is cheesemaking?"

result = rag_answer(question)

print("Q:", result["question"])
print("A:", result["answer"])
print("Confidence:", round(result["score"], 4))
print("\nRetrieved sources (chunk previews):")
for i, d in enumerate(result["source_documents"], 1):
    print(f"  [{i}] {d.page_content[:120].strip()}...")

In [ ]:
# Try a few more questions
for q in [
    "What is a computer virus?",
    "Who wrote the play Hamlet?",
    "What is machine learning?",
]:
    r = rag_answer(q)
    print(f"Q: {q}\nA: {r['answer']}  (score={r['score']:.3f})\n")

## 🎁 Bonus — a generative `RetrievalQA` chain
To use LangChain's `RetrievalQA` as described in the challenge, we need a **generative** model.
Here we swap in `google/flan-t5-base` (a `text2text-generation` model) so the challenge's
`RetrievalQA.from_chain_type` pattern works end-to-end and produces free-form answers instead of
extracted spans.

In [ ]:
from transformers import AutoModelForSeq2SeqLM

try:
    from langchain_huggingface import HuggingFacePipeline
except ImportError:
    from langchain_community.llms import HuggingFacePipeline

try:
    from langchain.chains import RetrievalQA
except ImportError:
    from langchain_classic.chains import RetrievalQA

gen_id = "google/flan-t5-base"
gen_tokenizer = AutoTokenizer.from_pretrained(gen_id)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(gen_id)

gen_pipe = pipeline(
    "text2text-generation",
    model=gen_model,
    tokenizer=gen_tokenizer,
    max_length=256,
    do_sample=False,
)
llm = HuggingFacePipeline(pipeline=gen_pipe)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
)

out = qa.invoke({"query": "What is cheesemaking?"})
print("Answer:", out["result"])
print("\nSources used:", len(out["source_documents"]))

## ✅ Conclusion
We built a complete **RAG** system:

- Loaded and cleaned `databricks/databricks-dolly-15k` into LangChain `Document`s.
- Chunked the text with `RecursiveCharacterTextSplitter` (1000 / 150 overlap).
- Embedded chunks with `all-MiniLM-L6-v2` and indexed them in **FAISS**.
- Retrieved relevant context and answered questions two ways:
  - **Extractive** QA with `Intel/dynamic_tinybert` (answer spans).
  - **Generative** QA with `flan-t5-base` via LangChain's `RetrievalQA` chain.

**Key takeaway:** match the chain to the model — extractive QA models answer from a supplied context
directly, while `RetrievalQA` expects a generative LLM.
